# ECG-FM Embedding Playground
Extracts embeddings from MIMIC-IV ECG records using the ECG Foundation Model (Meta AI).

**Pipeline:**
1. Load config + model
2. Probe output shape to confirm `hidden_dim`
3. Run batched GPU extraction
4. Save embeddings to parquet

In [ ]:
import pandas as pd
import os
import sys
import json
import importlib
from pathlib import Path
import numpy as np
import torch

# Notebook is in notebooks/, so repo root is parent
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "src" / "models" / "ecg_fm_playground_params.json"

In [ ]:
import src.models.ecg_fm_playground as efp
importlib.reload(efp)

print("Module loaded:", efp.__file__)

## 1. Load config and model

In [ ]:
cfg = efp.load_config(CONFIG_PATH)
print(json.dumps(cfg, indent=2))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
model = efp.load_ecgfm_model(cfg["model"]["checkpoint_path"], device)
print("Model loaded successfully.")

## 2. Probe output shape

Run this **before** extraction to confirm `hidden_dim` and update `embedding_dim` in the config if needed.
- If `hidden_dim = 256` → `embedding_dim = 512` ✔️
- If `hidden_dim = 512` → `embedding_dim = 1024` → update the JSON

In [ ]:
hidden_dim = efp.probe_model_output(model, device)

configured_dim = cfg["model"]["embedding_dim"]
actual_dim = hidden_dim * 2

## 3. Build subject/path list

Expects a DataFrame with at minimum: `subject_id`, `study_id`, `ecg_path`
where `ecg_path` is the WFDB base-path (no `.hea`/`.dat` extension).

In [ ]:
# --- Load your ECG manifest here ---
# Example: load from MIMIC-IV-ECG record list
BASE_DIR = Path(cfg["paths"]["base_records_dir"])

# Uncomment and adjust to your manifest:
# record_df = pd.read_csv(BASE_DIR / "record_list.csv")
# record_df["ecg_path"] = record_df["path"].apply(lambda p: str(BASE_DIR / p))

# --- Quick smoke-test with a single record ---
# Replace with a real path from your MIMIC-IV-ECG mount:
test_path = str(BASE_DIR / "p10" / "p10000032" / "s12345678" / "12345678")
record_df = pd.DataFrame({
    "subject_id": [10000032],
    "study_id": [12345678],
    "ecg_path": [test_path],
})

print(f"{len(record_df)} records to process")
record_df.head()

## 4. Run batched embedding extraction

In [ ]:
embs = efp.extract_embeddings_batched(
    model=model,
    signal_paths=record_df["ecg_path"].tolist(),
    device=device,
    batch_size=32,
    n_channels=12,
    target_samples=5000,
    segment_split=True,   # two 2500-sample halves → concatenated 512-dim
)

print(f"Embedding matrix shape: {embs.shape}")
print(f"  rows = records, cols = embedding dimensions")

## 5. Attach to DataFrame and save

In [ ]:
emb_cols = [f"emb_{i}" for i in range(embs.shape[1])]
emb_df = pd.DataFrame(embs, columns=emb_cols)

result_df = pd.concat(
    [record_df.reset_index(drop=True), emb_df],
    axis=1
)

print(result_df.shape)
result_df.head()

In [ ]:
OUT_PATH = REPO_ROOT / "data" / "processed" / "ecg_embeddings.parquet"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
result_df.to_parquet(OUT_PATH, index=False)
print(f"Saved → {OUT_PATH}")

## 6. GPU cleanup

In [ ]:
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU memory released.")